In [3]:
import os
root = "/kaggle/input/datasets/maryclauds/combined-final-final-dataset"
for dirpath, dirs, files in os.walk(root):
    depth = dirpath[len(root):].count(os.sep)
    if depth <= 2:
        print(f"{'  '*depth}{os.path.basename(dirpath) or root}/  ({len(files)} files, e.g. {files[:2]})")

combined-final-final-dataset/  (0 files, e.g. [])
  combined_final_final_dataset/  (1000 files, e.g. ['capture_2_35.jpg', 'capture_2_10.jpg'])


In [4]:
import os
import sys

# Confirm vision-common actually mounted, and under what exact folder name -
# Kaggle sometimes normalizes the slug slightly differently than you typed.
print(os.listdir('/kaggle/input'))
sys.path.append('/kaggle/input/datasets/maryclauds/vision-common-v8')

from vision_common_v8 import (
    rectify_workspace,
    detect_marker,
    extract_shape_features,
    find_valid_contours,
    compute_contour_features,
)


['datasets']


In [5]:
import os, glob, random, shutil, cv2
import vision_common_v8 as vc   # use whichever filename you have in the notebook

SRC = "/kaggle/input/datasets/maryclauds/combined-final-final-dataset"
OUT = "/kaggle/working/yolo_ds"
USE_RECTIFY = True    # label AND save the rectified image, so boxes match the pixels YOLO sees
VAL_FRACTION = 0.2
random.seed(42)

classes = sorted(d for d in os.listdir(SRC) if os.path.isdir(os.path.join(SRC, d)))
cls_id = {c: i for i, c in enumerate(classes)}
print("classes:", cls_id)

for split in ("train", "val"):
    os.makedirs(f"{OUT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT}/labels/{split}", exist_ok=True)

skipped, kept = 0, 0
for c in classes:
    paths = sorted(p for ext in ("jpg", "jpeg", "png")
                   for p in glob.glob(f"{SRC}/{c}/**/*.{ext}", recursive=True))
    random.shuffle(paths)
    n_val = int(len(paths) * VAL_FRACTION)

    for i, p in enumerate(paths):
        img = cv2.imread(p)
        if img is None:
            skipped += 1; continue
        if USE_RECTIFY:
            img = vc.preprocess_frame(img)

        marker = vc.detect_marker(img)
        contours = vc.find_valid_contours(
            img, exclude_bbox=marker["bbox"] if marker else None)
        if not contours:
            skipped += 1; continue

        cnt = max(contours, key=cv2.contourArea)       # one shape per image
        x, y, w, h = cv2.boundingRect(cnt)
        H, W = img.shape[:2]
        line = f"{cls_id[c]} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}\n"

        split = "val" if i < n_val else "train"
        stem = f"{c}_{os.path.splitext(os.path.basename(p))[0]}"
        cv2.imwrite(f"{OUT}/images/{split}/{stem}.jpg", img)
        with open(f"{OUT}/labels/{split}/{stem}.txt", "w") as f:
            f.write(line)
        kept += 1

with open(f"{OUT}/data.yaml", "w") as f:
    f.write(f"path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n")
    for c, i in cls_id.items():
        f.write(f"  {i}: {c}\n")

print(f"kept {kept}, skipped {skipped}")

classes: {'combined_final_final_dataset': 0}
kept 950, skipped 50
